In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from configs.config_loader import Config
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)
notebook_config = Config("configs/app_notebook.yaml")


DOMAIN = "it"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name, new_config=notebook_config)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("bench") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("bench") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("bench") / "data_dir/domains" / "base"
BASE_CODE_DIR      = Path("bench") / "domains" / "base"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/it
生成数据目录：bench/data_dir/domains/it
生成代码目录：bench/domains/it
BASE 数据目录：bench/data_dir/domains/base


### 0. 加载 BASE 上下文

In [2]:
# 加载 BASE 域生成结果（作为 HR 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()

with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")

加载完成


### 1. 生成 IT Schema

In [3]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", base_schema)

print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

正在生成 it Schema...
------------------------------------------------------------


```json
{
    "it_account": {
        "ITAC001": {
            "it_account_ref": "ITAC001",
            "employee_ref": "EMP001",
            "account_type": "email",
            "account_identifier": "zhangsan@company.com",
            "bound_system": "exchange",
            "account_status": "active",
            "last_login_at": "2025-01-15 09:00:00",
            "password_changed_at": "2025-01-15 09:00:00",
            "expires_at": null,
            "created_at": "2025-01-15 09:00:00",
            "updated_at": "2025-01-15 09:00:00"
        }
    },

    "access_permission": {
        "ACPM001": {
            "access_permission_ref": "ACPM001",
            "employee_ref": "EMP001",
            "target_system": "gitlab",
            "permission_level": "write",
            "resource_scope": "project:myapp",
            "approval_request_ref": "IAPR001",
            "granted_by_ref": "EMP002",
         

### 2. 生成 IT Data Models

In [4]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt = f.read()

models_prompt = models_prompt.replace("{base_models}", base_models)
models_prompt = models_prompt.replace("{domain_schema}", domain_schema)

print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
models_response = ""
for chunk in llm.llm.stream(models_prompt):
    content = chunk.content
    models_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

models_code = extract_code_block(models_response)
is_valid, err = verify_python_syntax(models_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {err}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), models_code)
print(f"✓ Models 已保存到: {models_path}")

正在生成 it Data Models...
------------------------------------------------------------


```python
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

from bench.environment.db import DB
from generated.domains.BASE.models import OfficeDB


# --- Enums as Literal types ---

ITAccountType = Literal["email", "vpn", "gitlab", "jira", "confluence", "ldap", "sso", "other"]
ITAccountStatus = Literal["active", "inactive", "suspended", "deleted"]
BoundSystem = Literal["exchange", "gitlab", "jira", "confluence", "ldap", "okta", "other"]

AccessPermissionLevel = Literal["read", "write", "admin"]
AccessPermissionStatus = Literal["active", "revoked", "expired"]

ITApprovalResourceType = Literal["account", "permission", "hardware", "software", "network", "other"]
ITApprovalRequestType = Literal["create", "modify", "delete", "grant", "revoke"]
ITApprovalPriority = Literal["low", "medium", "high", "urgent"]
ITApprovalStatus = Literal["pending", "approved", "rejec

### 3. Schema/Model一致性检查

In [ ]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

正在验证 Schema 和 Model 的一致性...
我来逐一检查这三点。

## 检查过程

### 1. Data model 中的字段和类型与 Schema 定义一致

逐表对比：

- **it_account**: ✅ 一致
- **access_permission**: ✅ 一致
- **it_approval_request**: ✅ 一致
- **email_account**: ✅ 一致
- **vpn_account**: ✅ 一致
- **network_access_policy**: ✅ 一致
- **software_license**: ✅ 一致
- **software_installation**: ✅ 一致
- **hardware_asset**: ✅ 一致
- **hardware_assignment**: ✅ 一致
- **printer**: ✅ 一致
- **print_consumable**: ✅ 一致
- **cloud_storage_quota**: ✅ 一致
- **backup_task**: ✅ 一致
- **code_repository**: ✅ 一致
- **repo_access**: ✅ 一致
- **it_ticket**: ✅ 一致
- **it_ticket_comment**: ✅ 一致

### 2. Data model 中的字段和类型在不同位置保持一致

检查跨模型引用和类型一致性：

- `PrintConsumable.currency` 使用的是 `LicenseCurrency` 类型，而 Schema 中 `print_consumable` 的 `currency` 值为 `"CNY"`，这与 `LicenseCurrency = Literal["USD", "CNY", "EUR", "JPY"]` 兼容。但语义上，`PrintConsumable` 复用了 `LicenseCurrency` 这个类型名，虽然值域相同，命名不太准确。不过这属于"Data model 更加合理"的范畴——值域是正确的，只是类型别名复用。

- 其他跨表引用字段（如 `employee_ref`、`it_account_ref`、`hardware_asset_ref` 等）类型

### 4. 生成数据脚本（Faker，主流程）

In [15]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 IT 域的外键依赖（不含 BASE 表），推导生成顺序
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"IT 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

local_llm = LLMClient("gpt-5.4-0305-global", 128000, new_config=notebook_config)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

IT 域表生成顺序（拓扑排序，共 18 张表）：
   1. itaccounts
   2. accesspermissions
   3. itapprovalrequests
   4. emailaccounts
   5. vpnaccounts
   6. networkaccesspolicies
   7. softwarelicenses
   8. softwareinstallations
   9. hardwareassets
  10. hardwareassignments
  11. printers
  12. printconsumables
  13. cloudstoragequotas
  14. backuptasks
  15. coderepositories
  16. repoaccesses
  17. ittickets
  18. itticketcomments

FK 依赖映射：
  itaccounts: [('it_account_ref', 'it_accounts'), ('employee_ref', 'employees')]
  accesspermissions: [('access_permission_ref', 'access_permissions'), ('employee_ref', 'employees'), ('approval_request_ref', 'approval_requests'), ('granted_by_ref', 'granted_bies'), ('revoked_by_ref', 'revoked_bies')]
  itapprovalrequests: [('it_approval_request_ref', 'it_approval_requests'), ('requester_ref', 'requesters'), ('resource_ref', 'resources'), ('approver_ref', 'approvers'), ('related_ticket_ref', 'related_tickets')]
  emailaccounts: [('email_account_ref', 'email_accounts')

### 3.5 执行数据脚本，生成 db.json

In [17]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)

print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（IT 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ IT Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 18 张表
  - it_account: 296 条
  - access_permission: 225 条
  - it_approval_request: 150 条
  - email_account: 112 条
  - vpn_account: 83 条
  - network_access_policy: 150 条
  - software_license: 37 条
  - software_installation: 150 条
  - hardware_asset: 160 条
  - hardware_assignment: 149 条
  - printer: 10 条
  - print_consumable: 10 条
  - cloud_storage_quota: 94 条
  - backup_task: 37 条
  - code_repository: 75 条
  - repo_access: 150 条
  - it_ticket: 150 条
  - it_ticket_comment: 375 条
✓ 外键完整性验证通过

✓ IT Database 已保存到: bench/data_dir/domains/it/db.json


In [18]:
# Pydantic 模型验证：合并 BASE + IT 数据后做完整校验
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from bench.domains.it.models import ITDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = json.load(f)

merged_raw = {**base_data, **db_raw}

try:
    it_db = ITDB.model_validate(merged_raw)
    print("✓ ITDB 模型验证通过")
    print(it_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ ITDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_it_accounts': 296, 'num_access_permissions': 225, 'num_it_approval_requests': 150, 'num_email_accounts': 112, 'num_vpn_accounts': 83, 'num_network_access_policies': 150, 'num_software_licenses': 37, 'num_software_installations': 150, 'num_hardware_assets': 160, 'num_hardware_assignments': 149, 'num_printers': 10, 'num_print_consumables': 10, 'num_cloud_storage_quotas': 94, 'num_backup_tasks': 37, 'num_code_repositories': 75, 'num_repo_accesses': 150, 'num_it_tickets': 150, 'num_it_ticket_comments': 375}


### 4. 生成 IT Tools

In [20]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# IT 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
it_function_description = """
Below is a PER-TABLE mandatory checklist. You MUST implement every function listed.
Use the EXACT function names specified. Do NOT skip any item.

--- BASE tools (already in OfficeTools, DO NOT reimplement) ---
SKIP: employee, department, team, position, role, permission, mentorship, org_config, notification
  These are all handled by the inherited OfficeTools class.

--- IT 工单 (IT Service Desk) ---
TABLE: it_ticket
  READ:  get_it_ticket(ticket_ref: str) -> ITTicket
  READ:  get_employee_it_tickets(employee_ref: str, status: Optional[str] = None) -> List[ITTicket]  # 按 created_at 降序
  READ:  list_it_tickets(status: Optional[str] = None, priority: Optional[str] = None, category: Optional[str] = None) -> List[ITTicket]
  WRITE: create_it_ticket(requester_employee_ref, ticket_type, priority, category, subject, description, assignee_employee_ref=None) -> ITTicket
  WRITE: update_it_ticket_status(ticket_ref, new_status, resolution=None, assignee_employee_ref=None) -> ITTicket  # 关闭时必须提供 resolution
  WRITE: assign_it_ticket(ticket_ref, assignee_employee_ref) -> ITTicket

--- 账号管理 (Account Management) ---
TABLE: it_account
  READ:  get_it_account(it_account_ref: str) -> ITAccount
  READ:  get_employee_it_accounts(employee_ref: str) -> List[ITAccount]  # 所有账号类型
  WRITE: create_it_account(employee_ref, account_type, account_name, systems=None, notes='') -> ITAccount
  WRITE: update_it_account_status(it_account_ref, new_status, reason='') -> ITAccount  # active/suspended/disabled/terminated

TABLE: access_permission
  READ:  get_access_permission(access_permission_ref: str) -> AccessPermission
  READ:  get_employee_access_permissions(employee_ref: str, status: Optional[str] = None) -> List[AccessPermission]
  WRITE: request_access_permission(employee_ref, target_system, permission_level, justification, expires_at=None) -> AccessPermission  # 创建待审批记录
  WRITE: approve_access_permission(access_permission_ref, approver_employee_ref, notes='') -> AccessPermission
  WRITE: revoke_access_permission(access_permission_ref, reason) -> AccessPermission

TABLE: it_approval_request
  READ:  get_it_approval_request(it_approval_request_ref: str) -> ITApprovalRequest
  READ:  get_pending_it_approvals(approver_employee_ref: str) -> List[ITApprovalRequest]
  READ:  get_employee_it_approval_requests(requester_employee_ref: str) -> List[ITApprovalRequest]
  WRITE: submit_it_approval_request(requester_employee_ref, resource_type, resource_ref, subject, description, approver_employee_ref) -> ITApprovalRequest
  WRITE: approve_it_approval_request(it_approval_request_ref, approver_employee_ref, notes='') -> ITApprovalRequest
  WRITE: reject_it_approval_request(it_approval_request_ref, approver_employee_ref, reason) -> ITApprovalRequest

--- 邮件管理 (Email Management) ---
TABLE: email_account
  READ:  get_email_account(email_account_ref: str) -> EmailAccount
  READ:  get_employee_email_accounts(employee_ref: str) -> List[EmailAccount]
  WRITE: create_email_account(employee_ref, email_address, email_type, total_quota_mb=5120) -> EmailAccount
  WRITE: update_email_account_quota(email_account_ref, new_total_quota_mb) -> EmailAccount
  WRITE: update_email_account_status(email_account_ref, new_status, reason='') -> EmailAccount

--- 网络管理 (Network Management) ---
TABLE: vpn_account
  READ:  get_vpn_account(vpn_account_ref: str) -> VPNAccount
  READ:  get_employee_vpn_accounts(employee_ref: str) -> List[VPNAccount]
  WRITE: create_vpn_account(employee_ref, protocol, server_address, expires_at=None) -> VPNAccount
  WRITE: update_vpn_account_status(vpn_account_ref, new_status, reason='') -> VPNAccount  # active/expired/suspended/terminated
  WRITE: renew_vpn_account(vpn_account_ref, new_expires_at) -> VPNAccount

TABLE: network_access_policy
  READ:  get_network_access_policy(network_access_policy_ref: str) -> NetworkAccessPolicy
  READ:  get_employee_network_access_policies(employee_ref: str) -> List[NetworkAccessPolicy]

--- 软件管理 (Software Management) ---
TABLE: software_license
  READ:  get_software_license(software_license_ref: str) -> SoftwareLicense
  READ:  list_software_licenses(software_name: Optional[str] = None) -> List[SoftwareLicense]
  READ:  get_expiring_software_licenses(before_date: str) -> List[SoftwareLicense]  # expires_at <= before_date

TABLE: software_installation
  READ:  get_software_installation(software_installation_ref: str) -> SoftwareInstallation
  READ:  get_employee_software_installations(employee_ref: str) -> List[SoftwareInstallation]
  READ:  get_license_installations(software_license_ref: str) -> List[SoftwareInstallation]
  WRITE: create_software_installation(employee_ref, software_name, version, software_license_ref=None) -> SoftwareInstallation  # 检查 license used_seats < total_seats
  WRITE: update_software_installation_status(software_installation_ref, new_status) -> SoftwareInstallation  # installed/uninstalled/pending/failed

--- 硬件管理 (Hardware Management) ---
TABLE: hardware_asset
  READ:  get_hardware_asset(hardware_asset_ref: str) -> HardwareAsset
  READ:  list_hardware_assets(asset_type: Optional[str] = None, status: Optional[str] = None) -> List[HardwareAsset]

TABLE: hardware_assignment
  READ:  get_hardware_assignment(hardware_assignment_ref: str) -> HardwareAssignment
  READ:  get_employee_hardware_assignments(employee_ref: str, status: Optional[str] = None) -> List[HardwareAssignment]
  READ:  get_asset_assignment_history(hardware_asset_ref: str) -> List[HardwareAssignment]
  WRITE: assign_hardware_to_employee(hardware_asset_ref, employee_ref, notes='') -> HardwareAssignment  # 检查资产状态为 available
  WRITE: return_hardware_from_employee(hardware_assignment_ref, condition_notes='') -> HardwareAssignment  # 更新资产状态为 available

--- 打印管理 (Printing Management) ---
TABLE: printer
  READ:  get_printer(printer_ref: str) -> Printer
  READ:  list_printers(status: Optional[str] = None) -> List[Printer]

TABLE: print_consumable
  READ:  get_print_consumable(print_consumable_ref: str) -> PrintConsumable
  READ:  list_print_consumables(consumable_type: Optional[str] = None) -> List[PrintConsumable]
  READ:  get_low_stock_consumables() -> List[PrintConsumable]  # current_stock < min_stock_threshold

--- 存储管理 (Storage Management) ---
TABLE: cloud_storage_quota
  READ:  get_cloud_storage_quota(cloud_storage_quota_ref: str) -> CloudStorageQuota
  READ:  get_employee_cloud_storage_quotas(employee_ref: str) -> List[CloudStorageQuota]
  WRITE: update_cloud_storage_quota(cloud_storage_quota_ref, new_total_quota_gb, reason='') -> CloudStorageQuota
"""

with open(BASE_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", it_function_description)

print(f"正在生成 {DOMAIN} Tools...")
print("-" * 60)
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 it Tools...
------------------------------------------------------------


```python
"""Toolkit for the IT domain, extending OfficeTools with IT-specific operations."""
from typing import Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolType, is_tool
from generated.domains.BASE.tools import OfficeTools
from bench.domains.it.models import (
    ITDB,
    ITAccount,
    AccessPermission,
    ITApprovalRequest,
    EmailAccount,
    VPNAccount,
    NetworkAccessPolicy,
    SoftwareLicense,
    SoftwareInstallation,
    HardwareAsset,
    HardwareAssignment,
    Printer,
    PrintConsumable,
    CloudStorageQuota,
    BackupTask,
    CodeRepository,
    RepoAccess,
    ITTicket,
    ITTicketComment,
)


class ITTools(OfficeTools):
    """Toolkit for the IT domain, extending OfficeTools with IT-specific operations."""

    db: ITDB

    def __init__(self, db: ITDB) -> None:
        """
        Initialize the ITTools too

### 5. 生成 Tasks（tasks.json）

结构驱动 pipeline（域无关）：ToolDAGParser → StateMachineExtractor → TaskPatternEnumerator → TaskInstantiator → LLM 场景包装。
难度由 task pattern 结构自动决定，不依赖 purposes。

In [ ]:
from synthesis.task_gen import TaskGenerationPipeline

with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    models_code = f.read()
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    domain_db = json.load(f)

pipeline = TaskGenerationPipeline(
    domain=DOMAIN,
    tools_code=tools_code,
    models_code=models_code,
    domain_db=domain_db,
    base_db=base_data,
    llm=llm,
)
tasks = pipeline.run()

tasks_path = GENERATED_DATA_DIR / "tasks.json"
with open(tasks_path, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)
print(f"已生成 {len(tasks)} 个任务 → {tasks_path}")

In [ ]:
# 任务已在上方 TaskGenerationPipeline.run() 中生成并保存

    task_prompt = task_prompt_template
    task_prompt = task_prompt.replace("{tool_code}", tools_code)
    task_prompt = task_prompt.replace("{base_content}", base_data_raw)
    task_prompt = task_prompt.replace("{domain_content}", domain_content_raw)
    task_prompt = task_prompt.replace("{purpose}", purpose)

    for attempt in range(MAX_RETRIES):
        task_response = llm.generate(task_prompt)
        task_text = extract_code_block(task_response)
        is_valid, task_data = verify_json_structure(task_text)
        if is_valid:
            break
        print(f"  ⚠️  第 {attempt+1} 次尝试失败：{task_data}")
        time.sleep(1)

    print(task_text)
    if is_valid:
        if isinstance(task_data, dict):
            task_data = [task_data]
        all_tasks.extend(task_data)
        print(f"✓ 生成成功，共 {len(task_data)} 个任务")
    else:
        print(f"✗ JSON 解析失败（已重试 {MAX_RETRIES} 次）: {task_data}")
    print('='*60)

tasks_json_text = json.dumps(all_tasks, ensure_ascii=False, indent=2)
tasks_path = GENERATED_DATA_DIR / "tasks.json"
save_generated_file(str(tasks_path), tasks_json_text)
print(f"\n✓ 共生成 {len(all_tasks)} 个任务，已保存到: {tasks_path}")

### 6. 任务质量验证

自动检查：
1. **ID 合法性**：actions.arguments 中所有 `*_ref` 值是否存在于 base_db 或 domain_db
2. **结构覆盖度**：5 种 pattern 的分布（guarded-write / trap / submit / cross-table / multi-entity）
3. **WRITE 工具覆盖率**：目标 ≥ 80%
4. **员工均匀度**：employee_ref 的变异系数 CV（目标 ≤ 0.3）
5. **WRITE 任务占比**：目标 ≥ 60%
6. **难度分布**：各 L1-L5 任务数量统计

In [ ]:
from synthesis.task_gen import verify_tasks, ToolDAGParser
import re as _re

with open(GENERATED_DATA_DIR / "tasks.json", 'r', encoding='utf-8') as f:
    tasks = json.load(f)
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    domain_db = json.load(f)
with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()

dag = ToolDAGParser().parse(tools_code)
metrics = verify_tasks(tasks, dag, base_data)

# 合并所有合法 ID（base + domain）
all_valid_ids = set()
for table_data in {**base_data, **domain_db}.values():
    if isinstance(table_data, dict):
        all_valid_ids.update(table_data.keys())

# ---- 1. ID 合法性检查 ----
id_errors = []
for task in tasks:
    task_id = task.get('id', '?')
    for action in task.get('evaluation_criteria', {}).get('actions', []):
        for arg_key, arg_val in action.get('arguments', {}).items():
            if arg_key.endswith('_ref') and isinstance(arg_val, str):
                if arg_val not in all_valid_ids:
                    id_errors.append(f"任务 {task_id} | 工具 {action['name']} | {arg_key}={arg_val} → 不存在于任何表")
print("=" * 60)
print("【ID 合法性检查】")
if not id_errors:
    print("✓ 所有 *_ref 参数均为合法 ID")
else:
    print(f"❌ 发现 {len(id_errors)} 处无效 ID：")
    for e in id_errors[:20]:
        print(f"  - {e}")

# ---- 2. 结构覆盖度（Pattern 分布）----
print("\n=== 结构覆盖度（Pattern 分布）===")
for pattern, count in sorted(metrics.get("pattern_coverage", {}).items()):
    print(f"  {pattern}: {count}")

# ---- 3. WRITE 工具覆盖率 ----
wc = metrics.get("write_tool_coverage")
if wc is not None:
    print(f"\nWRITE 工具覆盖率: {wc:.1%} {'✓' if wc >= 0.8 else '⚠️'}（目标 ≥ 80%）")
    all_write_tools = set(_re.findall(r'@is_tool\(ToolType\.WRITE\)\s*def (\w+)', tools_code))
    used_write_tools = {a['name'] for t in tasks for a in t.get('evaluation_criteria', {}).get('actions', []) if a.get('name') in dag.write_tools}
    uncovered_write = all_write_tools - used_write_tools
    if uncovered_write:
        print(f"  未覆盖 WRITE 工具: {sorted(uncovered_write)}")

# ---- 4. 员工均匀度 ----
eu = metrics.get("employee_uniformity")
if eu:
    print(f"员工均匀度 (CV): {eu['cv']:.2f} {'✓' if eu['cv'] <= 0.3 else '⚠️'}（目标 ≤ 0.3）")

# ---- 5. WRITE 任务占比 ----
wr = metrics.get("write_task_ratio")
if wr is not None:
    print(f"WRITE 任务占比: {wr:.1%} {'✓' if wr >= 0.6 else '⚠️'}（目标 ≥ 60%）")

# ---- 6. 难度分布 ----
print("\n=== 难度分布 ===")
total = len(tasks)
for level in ['L2', 'L3', 'L4', 'L5']:
    count = metrics.get("difficulty_distribution", {}).get(level, 0)
    print(f"  {level}: {count} ({count/total:.1%})")
print(f"  总计: {total}")